# Managed Hot-Load Walkthrough



This notebook walks through the exact flow you asked for:



1. Start a dummy-weight vLLM service.

2. Run inference and confirm the output is still dummy / nonsense.

3. Hot-load `Qwen3-1.7B-Base` and call the text generation API with `Ha Noi is ...`.

4. Hot-load `Qwen3-1.7B` and call the chat API.



The server process stays up across all steps. Only the weights change.



Pick the notebook kernel from `/home/anhvth8/vllm_projects/.venv` before running the code cells.

In [1]:
import json
import sys
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM

SKILL_SCRIPTS_DIR = Path('/home/anhvth8/vllm_projects/.agents/skills/vllm-managed-hotload/scripts')
if str(SKILL_SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SKILL_SCRIPTS_DIR))

from mangage_hotload_vllm import demo, describe_demo_config

def complete(prompt: str, max_tokens: int = 32) -> dict:
    return demo.post_json(
        '/v1/completions',
        {
            'model': demo.served_model_name,
            'prompt': prompt,
            'temperature': 0,
            'max_tokens': max_tokens,
        },
        timeout=120,
    )

def chat(prompt: str, max_tokens: int = 128) -> dict:
    return demo.post_json(
        '/v1/chat/completions',
        {
            'model': demo.served_model_name,
            'messages': [{'role': 'user', 'content': prompt}],
            'temperature': 0,
            'max_tokens': max_tokens,
            'chat_template_kwargs': {'enable_thinking': False},
        },
        timeout=120,
    )

print(json.dumps(describe_demo_config(), indent=2))

/home/anhvth8/vllm_projects/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "base_url": "http://127.0.0.1:8000",
  "served_model_name": "qwen3-1.7b",
  "dummy_load_format": "dummy",
  "base_model_path": "/home/anhvth8/ckpt/hf_models/Qwen/Qwen3-1.7B-Base",
  "chat_model_path": "/home/anhvth8/ckpt/hf_models/Qwen/Qwen3-1.7B",
  "helper_file": "/home/anhvth8/vllm_projects/.agents/skills/vllm-managed-hotload/scripts/mangage_hotload_vllm.py"
}


## Step 1. Host the model service with dummy weights



This starts the OpenAI-compatible server once with `--load-format dummy` and `--managed-weight-sync`.



Expected result:

- `/v1/models` returns the served model name.

- `/managed/status` and `/managed/world_size` return `200`.

- The model can answer requests, but the content is still dummy-weight garbage.

In [2]:
print(json.dumps(demo.start_dummy_service(), indent=2))
print(json.dumps(demo.models(), indent=2))
print(json.dumps(demo.world_size(), indent=2))

{
  "ok": true,
  "managed_weight_sync": true,
  "prefix": "/managed",
  "dev_mode": true,
  "served_model_names": [
    "qwen3-1.7b"
  ],
  "load_format": "dummy",
  "weight_transfer_config": {
    "backend": "ipc"
  },
  "sleep_mode_enabled": true,
  "is_sleeping": true,
  "world_size": 2
}
{
  "object": "list",
  "data": [
    {
      "id": "qwen3-1.7b",
      "object": "model",
      "created": 1778174229,
      "owned_by": "vllm",
      "root": "/home/anhvth8/ckpt/hf_models/Qwen/Qwen3-1.7B",
      "parent": null,
      "max_model_len": 4096,
      "permission": [
        {
          "id": "modelperm-8f2e16524670877f",
          "object": "model_permission",
          "created": 1778174229,
          "allow_create_engine": false,
          "allow_sampling": true,
          "allow_logprobs": true,
          "allow_search_indices": false,
          "allow_view": true,
          "allow_fine_tuning": false,
          "organization": "*",
          "group": null,
          "is_blocking"

In [ ]:
dummy_response = chat('Explain SVD using the smallest possible 2D example. Keep it short.', max_tokens=128)
print(json.dumps(dummy_response, indent=2))
print('\nExpected shape of the result: dummy output, repeated tokens, or nonsense before any real weight transfer.')

## Step 2. Load `Qwen3-1.7B-Base` and use the text generation API



This pushes base-model weights into the already-running server. After that, use `/v1/completions` with a plain prompt.



Suggested prompt: `Ha Noi is`



Expected result:

- The server process is unchanged.

- The completion should look like ordinary next-token text generation, for example a continuation about Hanoi being the capital of Vietnam.

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    dtype=torch.bfloat16,
    trust_remote_code=True,
 )
print(demo.push(base_model))

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 13122.15it/s]


Transferred in-memory model weights from the notebook process.
Parameters sent: 310
Target devices: [0, 1]
init_weight_transfer: {"ok": true, "initialized": true}
prepare_weight_update: {"ok": true, "steps": [{"step": "pause", "ok": true}, {"step": "sleep", "level": 2, "ok": true}, {"step": "wake_weights", "ok": true}]}
update_weights: {"message": "Weights updated"}
finish_weight_update: {"ok": true, "steps": [{"step": "wake_kv_cache", "ok": true}, {"step": "resume", "ok": true}]}


In [ ]:
demo.sleep(1)

{'ok': True, 'level': 1, 'sleeping': True}

In [ ]:
demo.wake()

{'ok': True, 'woke': True, 'tags': None}

In [ ]:
base_completion = demo.complete('Ha Noi is', max_tokens=32)
print(json.dumps(base_completion, indent=2))
print('\nExpected shape of the result: a plain text continuation instead of chat formatting.')

AttributeError: 'ManagedHotloadDemo' object has no attribute 'complete'

## Step 3. Load `Qwen3-1.7B` and use the chat API



Now push the chat-aligned checkpoint into the same live server and switch back to `/v1/chat/completions`.



Expected result:

- The answer should look like an assistant response, not raw text continuation.

- This confirms the same server can be reused while swapping checkpoints in place.

In [ ]:
chat_model = AutoModelForCausalLM.from_pretrained(
    CHAT_MODEL_PATH,
    dtype=torch.bfloat16,
    trust_remote_code=True,
 )
print(demo.push(chat_model))

Loading tokenizer from /home/anhvth8/ckpt/hf_models/Qwen/Qwen3-1.7B
Rendered chat prompt:
<|im_start|>user
Explain SVD using the smallest possible 2D example. Keep it short.<|im_end|>
<|im_start|>assistant
<think>

</think>


Loading model from /home/anhvth8/ckpt/hf_models/Qwen/Qwen3-1.7B on cuda:0
Initializing managed IPC weight transfer
{'ok': True, 'initialized': True}
Preparing server for weight update
{'ok': True, 'steps': [{'step': 'pause', 'ok': True}, {'step': 'sleep', 'level': 2, 'ok': True}, {'step': 'wake_weights', 'ok': True}]}
Sending model weights via CUDA IPC to target devices 0,1
Finishing server weight update
{'ok': True, 'steps': [{'step': 'wake_kv_cache', 'ok': True}, {'step': 'resume', 'ok': True}]}
Generating after weight transfer:
Sure! Here's a short explanation of SVD using a 2D example:

**SVD (Singular Value Decomposition)** is a way to break down a matrix into three simpler matrices.  
For a 2D example, imagine a 2x2 matrix. SVD decomposes it into three matri

In [10]:
chat_result = demo.chat('Write one short sentence about Hanoi.', max_tokens=64)
print(json.dumps(chat_result, indent=2))
print('\nExpected shape of the result: a coherent assistant-style reply in chat format.')

{
  "id": "chatcmpl-a6b6e1adf85ea889",
  "object": "chat.completion",
  "created": 1778172842,
  "model": "qwen3-1.7b",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": ".WriteHeader\n.WriteHeader\n.WriteHeader\n.WriteHeader\n.WriteHeader\n.WriteHeader\n.WriteHeader\n.WriteHeader\n.WriteHeader\n.WriteHeader\n.WriteHeader\n.WriteHeader\n.WriteHeader\n.WriteHeader\n.WriteHeader\n.WriteHeader\n.WriteHeader\n.WriteHeader\n.WriteHeader\n.WriteHeader\n.WriteHeader\n.WriteHeader\n.WriteHeader\n.WriteHeader\n.WriteHeader\n.WriteHeader\n.WriteHeader\n.WriteHeader\n.WriteHeader\n.WriteHeader\n.WriteHeader\n.WriteHeader\n",
        "refusal": null,
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": [],
        "reasoning": null
      },
      "logprobs": null,
      "finish_reason": "length",
      "stop_reason": null,
      "token_ids": null
    }
  ],
  "service_tier": null,
  "system_

In [ ]:
print(json.dumps(demo.pause(), indent=2))
print(json.dumps(demo.sleep(level=1), indent=2))
print(json.dumps(demo.wake(tags=['weights', 'kv_cache']), indent=2))
print(json.dumps(demo.resume(), indent=2))

## Optional offload without stopping

Use the next cell if you want to park the server in-process instead of killing it outright. This demonstrates the managed control surface directly from the notebook: pause generation, offload memory, wake the engine back up, then resume.

## Optional cleanup



Run the next cell if you want to stop the notebook-started server and free the GPUs.

In [ ]:
print(demo.stop())